In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import ast
from scipy.stats import norm

# Analysis of the results of the CSV'S resulting from the sim

In this notebook it will be shown the process of analyzing the resulting sim data and stablishing a pipeline which will lead to a .py projectwhich will automatically run after the GAMA sim and be able to extract the results via graphics and similar

First there is a lot of csvs. we need to joint them all into one

In [10]:
BASE_SIM_DIR = "20260406_110502"

DESCRIPTION = {}

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/description.txt", "r") as f:
    for line in f:
        parts = line.strip().split(" ", 1)  # split only on first space
        if len(parts) == 2:
            DESCRIPTION[parts[0]] = parts[1]

if DESCRIPTION.get("line") == "401":
    stops_list_toA = [2646, 562, 4090, 4188, 5939, 3995, 3991, 3903, 2539, 2535, 1929, 4032, 5360, 82, 5727]
    stops_list_toB = [5727, 81, 1167, 2538, 3929, 3990, 3994, 5938, 4187, 2964, 649, 2646]
    to_A = "Cibeles"
    to_B = "La Peseta"



#HARDCODDED NUMBER OF SIMS
N_SIMS = 1000

columns = ["simulation_id", "bus_code", "travels_completed","stop_num", "direction", "current_time","timeslot","boards","residual_boards","dropoffs", "occupation", "time_between_stops"]

joint_dataframe_results = pd.DataFrame(columns=columns)

dataframes_to_join_results = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/RESULTS/log_Simulation {iteration}.csv", skiprows = 1,
            names = columns, encoding="utf-8")
    dataframes_to_join_results.append(new_dataframe)



joint_dataframe_results = pd.concat(data for data in dataframes_to_join_results)


In [11]:
N_SIMS = 1000


columns = ["simulation_id", "bus_code","travels_completed", "stop_num", "direction","timeslot", "event_code", "time_event","param_event_1", "param_event_2"]

joint_dataframe_events = pd.DataFrame(columns=columns)

dataframes_to_join_events = []
for iteration in range(0,N_SIMS):
    try:
        new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/EVENTS/log_Simulation {iteration}.csv",skiprows = 1,
                             names = columns, encoding="utf-8")
        dataframes_to_join_events.append(new_dataframe)
    except:
        pass


joint_dataframe_events = pd.concat(data for data in dataframes_to_join_events)

In [12]:
columns = ["simulation_id", "bus_code","travels_completed", "stop_num", "timeslot", "current_time","list_passengers_time"]

joint_dataframe_passengers = pd.DataFrame(columns=columns)

dataframes_to_join_passengers = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PASSENGERS/log_Simulation {iteration}.csv", skiprows = 1,
                names=columns, encoding="utf-8")
    dataframes_to_join_passengers.append(new_dataframe)



joint_dataframe_passengers = pd.concat(data for data in dataframes_to_join_passengers)



joint_dataframe_passengers["list_passengers_time"] = (
    joint_dataframe_passengers["list_passengers_time"]
    .apply(ast.literal_eval)
)


In [13]:
known_occupation_values_toA = pd.read_csv(f"{os.getcwd()}/known_data/NORMAL_OCCUPATION_toA.csv")
known_occupation_values_toB = pd.read_csv(f"{os.getcwd()}/known_data/NORMAL_OCCUPATION_toB.csv")
known_times_stops_toA = pd.read_csv(f"{os.getcwd()}/known_data/NORMAL_times_stops_toA.csv")
known_times_stops_toB = pd.read_csv(f"{os.getcwd()}/known_data/NORMAL_times_stops_toB.csv")

Once joined the data, its time to start making the graphics results andif well done, elevate the process to a .py code that accepts the ISODATE of the sim as a parameter, and build a directory which holds the graphics results.

## Graphic 1. Statistics of simulation Occupation VS REAL TIME occupation

In [14]:
# la idea de esto es generar gráficas a mansalva. La idea detrás ed esto es poder tener tantas gráficas como se guste y en la interfaz de usuario poder agrupar esas gráficas para que se den varios puntos de vista.


#idea 1. A parte de hacer gráficas para todos los timeslots, agrupar por hardcodded timestamps (Mañana[5], Hora punta[6,9], Valle de Mañana[10,13], Hora tarde[14,19], noche[20,23]
timeslots = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]



# Estadísticas generales. Medidas de Rendimiento. % de ocupación medio, general, por franja horaria, Enfrentamiento real vs teórico "normal" de ocupacion

In [15]:
#-------------------------------------------------------------------------
#-----------------------Imported from fussion gauss-----------------------
#-------------------------------------------------------------------------
import math

from math import sqrt

from math import log

from math import exp




threshold = 0.97

def bhattacharyya_distance(mean_1, mean_2, std_1, std_2):
    return 0.25*((mean_1 -mean_2)**2/(std_1**2 + std_2**2)) + 0.5*(log((std_1**2 + std_2**2)/(2*(std_1 * std_2))))





def bhattacharyya_coefficient(mean_1, mean_2, std_1, std_2):
    return (sqrt(2*(std_1*std_2/(std_1**2 + std_2**2))))*exp(-(((mean_1 -mean_2)**2)/(4*(std_1**2 + std_2**2))))


import numpy as np



def central_moment(x, r):
    mean = np.nanmean(x)
    return sum((xi - mean) ** r for xi in x) / len(x)


def kurtosis(x):
    return (central_moment(x, 4) / central_moment(x, 2) ** 2) -3

def skewness(x):
    return central_moment(x, 3) / central_moment(x, 2)**(3/2)


def is_normal_simplified(variables:list):


    if len(variables) == 0:
        return np.bool(False), [], 0, 0

    elif len(variables) < 50: #CASO Z score 1.96
        z_crit = 1.96
    elif len(variables) < 300: #CASO Z score 3.29
        z_crit = 3.29
    else:
        z_crit = "False"

    skew = skewness(variables)
    kurt = kurtosis(variables)


    n = len(variables)
    se_skew = math.sqrt(6.0 / n)
    se_kurt = math.sqrt(24.0 / n)

    mean = np.nanmean(variables)
    std_dev = math.sqrt(central_moment(variables, 2))
    z_skew = skew / se_skew
    z_kurt = kurt / se_kurt

    if z_crit != "False":
        approx_normal = (abs(z_skew) <= z_crit) and (abs(z_kurt) <= z_crit)

    else:
        approx_normal = (abs(skew) <= 2) and (abs(kurt) + 3 <= 7)

    return approx_normal, variables, mean, std_dev



In [16]:
 #KPI 1. % de ocupación medio. % de ocupacion por franja horaria. % de ocupación de cada autobús en acda una de sus franjas
MAX_CAPACITY = 80

def time_to_seconds(time_str, include_seconds: bool):
    if include_seconds:
        h, m, s = map(int, time_str.split(":"))
    else:
        h,m = map(int, time_str.split(":"))
        s = 0
    return h * 3600 + m * 60 + s


#La easy.
percentage_occupation_per_stop = 0


json_output_general_info = {}



data_toA = joint_dataframe_results[joint_dataframe_results["direction"] == "toA"]
data_toB = joint_dataframe_results[joint_dataframe_results["direction"] == "toB"]

json_output_general_info["Occupation_toA"] = (data_toA["occupation"].mean()/MAX_CAPACITY)*100
json_output_general_info["Occupation_toB"] = (data_toB["occupation"].mean()/MAX_CAPACITY)*100
json_output_general_info["Occupation"] = (joint_dataframe_results["occupation"].mean()/MAX_CAPACITY)*100

#extract also the logical buses that run the sim to save it in the general info
logic_buses = joint_dataframe_results["bus_code"].unique()
json_output_general_info["Logical_Buses"] = len(logic_buses)

#also add the total number of trips that were made
trips_counter = 0
for bus in logic_buses:
    data_per_bus = joint_dataframe_results[joint_dataframe_results["bus_code"] == bus]["travels_completed"]
    trips_counter += (max(data_per_bus) + 1)# +1 because travels_completed starts at 0

json_output_general_info["Total_Trips"] = trips_counter

#add also the mean wait time per passenger


#compute time will go brrrr
mean_time_awaital_passenger = np.nanmean(
    [t for data in joint_dataframe_passengers["list_passengers_time"] for t in data]
)

json_output_general_info["Mean_await_passengers"] = mean_time_awaital_passenger




#Empiezan las graficassss
#graficas de evolucion a lo largo del tiempon de valor de %Occupation
list_occupation_values_toA = []
list_occupation_values_toB = []
list_occupation = []
for timeslot in timeslots:
    data_toA_per_ts = data_toA[data_toA["timeslot"] == timeslot]
    data_toB_per_ts = data_toB[data_toB["timeslot"] == timeslot]
    data_general = joint_dataframe_results[joint_dataframe_results["timeslot"] == timeslot]
    list_occupation_values_toA.append((data_toA_per_ts["occupation"].mean()/MAX_CAPACITY)*100)
    list_occupation_values_toB.append((data_toB_per_ts["occupation"].mean()/MAX_CAPACITY)*100)
    list_occupation.append((data_general["occupation"].mean()/MAX_CAPACITY)*100)


plt.style.use('seaborn-v0_8-whitegrid')

def save_barcode(horizontal_data, vertical_data, horizontal_label, vertical_label, leyend, label, filename):
    x = np.arange(len(horizontal_data))
    plt.figure(figsize=(12,5))
    plt.bar(x, vertical_data, width=0.4, label=leyend)
    plt.xticks(x + 0.2, horizontal_data, rotation=90)
    plt.ylabel(vertical_label)
    plt.xlabel(horizontal_label)
    plt.title(label)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()


def compute_and_save_distribution_comparasion(mean_1,std_1, mean_2, std_2, direction, stopcode, timeslot):


    # Guard against zero or near-zero std (no valid distribution to plot)
    if std_1 <= 1e-9 or std_2 <= 1e-9:
        return

    # Bhattacharyya metrics
    bhatt_dist = bhattacharyya_distance(mean_1, mean_2, std_1, std_2)
    bhatt_coeff = bhattacharyya_coefficient(mean_1, mean_2, std_1, std_2)

    #cover both distributions
    x_min = min(mean_1 - 4 * std_1, mean_2 - 4 * std_2)
    x_max = max(mean_1 + 4 * std_1, mean_2 + 4 * std_2)
    x = np.linspace(x_min, x_max, 500)

    y1 = norm.pdf(x, mean_1, std_1)
    y2 = norm.pdf(x, mean_2, std_2)

    # Verdict based on threshold
    if bhatt_coeff >= threshold:
        verdict = "Fusionables ('Equivalentes')"
    else:
        verdict = "No fusionables (Distintas)"

    plt.figure(figsize=(12, 5))
    plt.plot(x, y1, color="blue", linewidth=2,
             label=f"Simulación (μ={mean_1:.2f}, σ={std_1:.2f})")
    plt.fill_between(x, y1, alpha=0.15, color="blue")

    plt.plot(x, y2, color="red", linewidth=2,
             label=f"Esperado (μ={mean_2:.2f}, σ={std_2:.2f})")
    plt.fill_between(x, y2, alpha=0.15, color="red")

    plt.xlabel("Ocupación del Autobus al paso por parada")
    plt.ylabel("Densidad de probabilidad")
    plt.title(f"Comparativa de distribuciones – parada {stopcode}, Franja {timeslot}")
    plt.legend(
        title=f"Distancia de Bhattacharyya ={bhatt_dist:.4f}  coeff={bhatt_coeff:.4f}\n→ {verdict}",
        loc="upper right"
    )
    plt.tight_layout()
    plt.savefig(
        f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/{direction}/stops/"
        f"occupation_comparaison_{stopcode}_per_timeslot_{timeslot}.png",
        dpi=300
    )
    plt.close()



def compute_and_save_distribution_comparasion_times(mean_1,std_1, mean_2, std_2, direction, stopcode, timeslot):


    # Guard against zero or near-zero std (no valid distribution to plot)
    if std_1 <= 1e-9 or std_2 <= 1e-9:
        return

    # Bhattacharyya metrics
    bhatt_dist = bhattacharyya_distance(mean_1, mean_2, std_1, std_2)
    bhatt_coeff = bhattacharyya_coefficient(mean_1, mean_2, std_1, std_2)

    #cover both distributions
    x_min = min(mean_1 - 4 * std_1, mean_2 - 4 * std_2)
    x_max = max(mean_1 + 4 * std_1, mean_2 + 4 * std_2)
    x = np.linspace(x_min, x_max, 500)

    y1 = norm.pdf(x, mean_1, std_1)
    y2 = norm.pdf(x, mean_2, std_2)

    # Verdict based on threshold
    if bhatt_coeff >= threshold:
        verdict = "Fusionables ('Equivalentes')"
    else:
        verdict = "No fusionables (Distintas)"

    plt.figure(figsize=(12, 5))
    plt.plot(x, y1, color="blue", linewidth=2,
             label=f"Simulación (μ={mean_1:.2f}, σ={std_1:.2f})")
    plt.fill_between(x, y1, alpha=0.15, color="blue")

    plt.plot(x, y2, color="red", linewidth=2,
             label=f"Esperado (μ={mean_2:.2f}, σ={std_2:.2f})")
    plt.fill_between(x, y2, alpha=0.15, color="red")

    plt.xlabel(f"Tiempo medio entre paradas {stopcode[0]}-{stopcode[1]} medido en segundos ")
    plt.ylabel("Densidad de probabilidad")
    plt.title(f"Comparativa de distribuciones – paradas {stopcode[0]}-{stopcode[1]}, Franja {timeslot}")
    plt.legend(
        title=f"Distancia de Bhattacharyya ={bhatt_dist:.4f}  coeff={bhatt_coeff:.4f}\n→ {verdict}",
        loc="upper right"
    )
    plt.tight_layout()
    plt.savefig(
        f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/{direction}/times/"
        f"time_between_stops_comparaison_{stopcode[0]}-{stopcode[1]}_per_timeslot_{timeslot}.png",
        dpi=300
    )
    plt.close()



save_barcode(timeslots, list_occupation_values_toA , "TimeSlots", "% de Ocupación",  f"Sentido {to_A}", "Ocupación del bus por franja horaria", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/occupation_per_timeslot_toA.png")

save_barcode(timeslots, list_occupation_values_toB , "TimeSlots", "% de Ocupación",  f"Sentido {to_B}", "Ocupación del bus por franja horaria", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/occupation_per_timeslot_toB.png")

save_barcode(timeslots, list_occupation , "TimeSlots", "% de Ocupación",  "Datos Generales", "Ocupación del bus por franja horaria",
             f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/general/occupation_per_timeslot_general.png")



stops_list_toA = [int(data) for data in stops_list_toA]
stops_list_toB = [int(data) for data in stops_list_toB]



json_stops_info_toA = {}
json_stops_info_toB = {}

for stopcode in stops_list_toA:
    #obtain the data from that stop and extract per timeslots its value
    list_per_ts_per_stop = []
    json_stops_info_toA[f"{stopcode}"] = {}

    data_stopcode = data_toA[data_toA["stop_num"] == stopcode]

    json_stops_info_toA[f"{stopcode}"]["Occupation_percentage_per_timeslot"] = {}

    json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"] = {}

    if stopcode == stops_list_toA[-1]:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            actual_percentage = (data_per_ts_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100
            list_per_ts_per_stop.append((actual_percentage, data_per_ts_per_stop["dropoffs"],0, 0,data_per_ts_per_stop["dropoffs"].mean()))
            json_stops_info_toA[f"{stopcode}"]["Occupation_percentage_per_timeslot"][timeslot] = actual_percentage
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot] = {}
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot]["boards"] = 0
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot]["unable_boards"] = 0
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot]["dropoffs"] = data_per_ts_per_stop["dropoffs"].mean()



        save_barcode(timeslots, [data[0] for data in list_per_ts_per_stop] , "TimeSlots", "% de Ocupación",  f"Stop {stopcode}", "Ocupación media por franja horaria de los autobuses", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/stops/occupation_per_timeslot_{stopcode}.png")

    else:

        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            actual_percentage = (data_per_ts_per_stop["occupation"].mean()/MAX_CAPACITY)*100
            list_per_ts_per_stop.append((actual_percentage, data_per_ts_per_stop["occupation"], data_per_ts_per_stop["boards"].mean(), data_per_ts_per_stop["residual_boards"].mean(), data_per_ts_per_stop["dropoffs"].mean()))
            json_stops_info_toA[f"{stopcode}"]["Occupation_percentage_per_timeslot"][timeslot] = actual_percentage
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot] = {}
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot]["boards"] = data_per_ts_per_stop["boards"].mean()
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot]["unable_boards"] = data_per_ts_per_stop["residual_boards"].mean()
            json_stops_info_toA[f"{stopcode}"]["Board_dropoff_per_timeslot"][timeslot]["dropoffs"] = data_per_ts_per_stop["dropoffs"].mean()

        save_barcode(timeslots, [data[0] for data in list_per_ts_per_stop] , "TimeSlots", "% de Ocupación",  f"Stop {stopcode}", "Ocupación media por franja horaria de los autobuses", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/stops/occupation_per_timeslot_{stopcode}.png")

    json_stops_info_toA[f"{stopcode}"]["Occupation_percentage"] = np.nanmean([data[0] for data in list_per_ts_per_stop])
    json_stops_info_toA[f"{stopcode}"]["mean_boards"] = np.nanmean([data[2] for data in list_per_ts_per_stop])
    json_stops_info_toA[f"{stopcode}"]["unable_boards"] = np.nanmean([data[3] for data in list_per_ts_per_stop])
    json_stops_info_toA[f"{stopcode}"]["mean_dropoffs"] = np.nanmean([data[4] for data in list_per_ts_per_stop])




    #Important KPI. Theoretical vs Real Distribution of Occupation per Timeslot per Stop
    for iter_ts in range(0, (timeslots[-1] - timeslots[0]) + 1):
        #Very important. Lets compare now the distribution of occupation in that stop VS the theoreticall distribution obtained in the statistical part
        normal_sim, vars_sim, mean_sim, std_sim = is_normal_simplified(list_per_ts_per_stop[iter_ts][1].tolist())

        #now, obtain the ts corresponding to this iteration
        timeslot = timeslots[iter_ts]
        #obtain the mean and std coirresponmding to this stop and this ts
        data_stop = known_occupation_values_toA[known_occupation_values_toA["Stop"] == stopcode]
        mean_known = data_stop[f"Mean_{timeslot}"].iloc[0]
        std_known = data_stop[f"std_{timeslot}"].iloc[0]

        compute_and_save_distribution_comparasion(mean_sim, std_sim, mean_known, std_known, "to_A", stopcode, timeslot)


    #Per stop. Compute the mean time of awaital per passenger(Normal and per timeslot)

        #first normal mean time of awaital
    data_passengers_in_stop = joint_dataframe_passengers[joint_dataframe_passengers["stop_num"] == stopcode]
    mean_time_awaital_passenger = np.nanmean(
    [t for data in data_passengers_in_stop["list_passengers_time"] for t in data]
    )

    json_stops_info_toA[f"{stopcode}"]["Mean_await_time_general"] = mean_time_awaital_passenger

    json_stops_info_toA[f"{stopcode}"]["Mean_await_time_per_timeslot"] = {}

    for timeslot in timeslots:
        data_passengers_timeslotted = data_passengers_in_stop[data_passengers_in_stop["timeslot"] == timeslot]
        mean_time_awaital_passenger_timeslotted = np.nanmean(
        [t for data in data_passengers_timeslotted["list_passengers_time"] for t in data]
        )
        json_stops_info_toA[f"{stopcode}"]["Mean_await_time_per_timeslot"][timeslot] = mean_time_awaital_passenger_timeslotted



for stopcode in stops_list_toB:
    #obtain the data from that stop and extract per timeslots its value
    list_per_ts_per_stop = []

    json_stops_info_toB[f"{stopcode}"] = {}
    json_stops_info_toB[f"{stopcode}"]["Occupation_percentage_per_timeslot"] = {}
    data_stopcode = data_toB[data_toB["stop_num"] == stopcode]
    if stopcode == stops_list_toB[-1]:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            actual_percentage = (data_per_ts_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100
            list_per_ts_per_stop.append((actual_percentage, data_per_ts_per_stop["dropoffs"]))
            json_stops_info_toB[f"{stopcode}"]["Occupation_percentage_per_timeslot"][timeslot] = actual_percentage

        save_barcode(timeslots, [data[0] for data in list_per_ts_per_stop] , "TimeSlots", "% de Ocupación",  f"Stop {stopcode}", "Ocupación media por franja horaria de los autobuses", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/stops/occupation_per_timeslot_{stopcode}.png")
    else:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            actual_percentage = (data_per_ts_per_stop["occupation"].mean()/MAX_CAPACITY)*100
            list_per_ts_per_stop.append((actual_percentage, data_per_ts_per_stop["occupation"]))
            json_stops_info_toB[f"{stopcode}"]["Occupation_percentage_per_timeslot"][timeslot] = actual_percentage

        save_barcode(timeslots, [data[0] for data in list_per_ts_per_stop] , "TimeSlots", "% de Ocupación",  f"Stop {stopcode}", "Ocupación media por franja horaria de los autobuses", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/stops/occupation_per_timeslot_{stopcode}.png")

    json_stops_info_toB[f"{stopcode}"]["Occupation_percentage"] = np.nanmean([data[0] for data in list_per_ts_per_stop])


    #Important KPI. Theoretical vs Real Distribution of Occupation per Timeslot per Stop
    for iter_ts in range(0, ((timeslots[-1] - timeslots[0]) + 1)):
        #Very important. Lets compare now the distribution of occupation in that stop VS the theoreticall distribution obtained in the statistical part
        normal_sim, vars_sim, mean_sim, std_sim = is_normal_simplified(list_per_ts_per_stop[iter_ts][1].tolist())

        #now, obtain the ts corresponding to this iteration
        timeslot = timeslots[iter_ts]
        #obtain the mean and std coirresponmding to this stop and this ts
        data_stop = known_occupation_values_toB[known_occupation_values_toB["Stop"] == stopcode]
        mean_known = data_stop[f"Mean_{timeslot}"].iloc[0]
        std_known = data_stop[f"std_{timeslot}"].iloc[0]

        compute_and_save_distribution_comparasion(mean_sim, std_sim, mean_known, std_known, "to_B", stopcode, timeslot)



        #Per stop. Compute the mean time of awaital per passenger(Normal and per timeslot)

        #first normal mean time of awaital
    data_passengers_in_stop = joint_dataframe_passengers[joint_dataframe_passengers["stop_num"] == stopcode]
    mean_time_awaital_passenger = np.nanmean(
    [t for data in data_passengers_in_stop["list_passengers_time"] for t in data]
    )

    json_stops_info_toB[f"{stopcode}"]["Mean_await_time_general"] = mean_time_awaital_passenger

    json_stops_info_toB[f"{stopcode}"]["Mean_await_time_per_timeslot"] = {}

    for timeslot in timeslots:
        data_passengers_timeslotted = data_passengers_in_stop[data_passengers_in_stop["timeslot"] == timeslot]
        mean_time_awaital_passenger_timeslotted = np.nanmean(
        [t for data in data_passengers_timeslotted["list_passengers_time"] for t in data]
        )
        json_stops_info_toB[f"{stopcode}"]["Mean_await_time_per_timeslot"][timeslot] = mean_time_awaital_passenger_timeslotted

#Now occupation per physical bus
unique_bus = joint_dataframe_results["bus_code"].unique()


input_json_route = f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/includes/INPUTS/input_data_buses.json"

with open(input_json_route, "r") as f:
    input_json = json.load(f)

new_bus_json_data = {}


for bus in unique_bus:

    trip_occupation_percentage = 0

    data_per_bus = joint_dataframe_results[joint_dataframe_results["bus_code"] == bus]
    events_per_bus = joint_dataframe_events[joint_dataframe_events["bus_code"] == bus]

    unique_trips = data_per_bus['travels_completed'].unique()
    unique_trips = [int(data) for data in unique_trips]

    new_bus_json_data[f"{bus}"] = {}

    #Loop over each trip
    for trip in unique_trips:

        events_per_unique_trip =events_per_bus[events_per_bus["travels_completed"] == trip]

        total_percentage_of_delays_per_trip = (len(events_per_unique_trip)/N_SIMS)*100
        mean_delay_in_minutes_per_trip = (pd.Series([time_to_seconds(data, True) for data in events_per_unique_trip["time_event"]]) - pd.Series([time_to_seconds(data, False) for data in events_per_unique_trip["param_event_1"]])).mean()



        data_inputs = input_json[str(bus)][trip]

        data_simulation_per_trip = data_per_bus[data_per_bus['travels_completed'] == trip]
        trip_occupation_percentage = (data_simulation_per_trip["occupation"].mean()/MAX_CAPACITY)*100

        events_per_trip_per_bus = events_per_bus[events_per_bus['travels_completed'] == trip]

        mean_time_of_trip = 0
        #Extract the mean ocupation for saving the plot

        list_mean_values = []
        direction = ""
        if int(data_inputs[0]) == stops_list_toA[0]:
            direction = "to_A"

            #extract the mean estimated time. Calculating the mean arrival time to stop stops_list_toA[-1] and the mean departure time from stops_list_toA[-1]
            data_per_trip_last_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toA[-1]]
            time_per_last_stop = data_per_trip_last_stop["current_time"]
            time_per_last_stop = [time_to_seconds(data, True) for data in time_per_last_stop]
            data_per_trip_first_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toA[0]]
            time_per_first_stop = data_per_trip_first_stop["current_time"]
            time_per_first_stop = [time_to_seconds(data, True) for data in time_per_first_stop]
            mean_time_of_trip = sum(time_per_last_stop)/len(time_per_last_stop) - sum(time_per_first_stop)/len(time_per_first_stop)

            for stop in stops_list_toA:
                data_per_trip_per_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stop]
                if stop != stops_list_toA[-1]:
                    list_mean_values.append( (data_per_trip_per_stop["occupation"].mean()/MAX_CAPACITY)*100)
                else:
                    list_mean_values.append( (data_per_trip_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100)


            save_barcode(stops_list_toA, list_mean_values , f"Paradas en sentido {to_A}", "% de Ocupación",  f"Ocupación del viaje nº: {trip}", f"Ocupación del autobús por parada para el bus: {bus} con salida estimada {data_inputs[1]}", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/mean_occupation_per_stop_toA_{bus}_{trip}.png")




        else:
            direction = "toB"
            #extract the mean estimated time. Calculating the mean arrival time to stop stops_list_toA[-1] and the mean departure time from stops_list_toA[-1]
            data_per_trip_last_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toB[-1]]
            time_per_last_stop = data_per_trip_last_stop["current_time"]
            time_per_last_stop = [time_to_seconds(data, True) for data in time_per_last_stop]
            data_per_trip_first_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toB[0]]
            time_per_first_stop = data_per_trip_first_stop["current_time"]
            time_per_first_stop = [time_to_seconds(data, True) for data in time_per_first_stop]
            mean_time_of_trip = sum(time_per_last_stop)/len(time_per_last_stop) - sum(time_per_first_stop)/len(time_per_first_stop)


            for stop in stops_list_toB:
                data_per_trip_per_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stop]
                if stop != stops_list_toB[-1]:
                    list_mean_values.append( (data_per_trip_per_stop["occupation"].mean()/MAX_CAPACITY)*100)
                else:
                    list_mean_values.append( (data_per_trip_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100)

            save_barcode(stops_list_toB, list_mean_values , f"Paradas en sentido {to_B}", "% de ocupación",  f"Ocupación del viaje nº: {trip}", f"Ocupación del autobús por parada para el bus: {bus} con salida estimada {data_inputs[1]}", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/mean_occupation_per_stop_toB_{bus}_{trip}.png")


        #Now data_inputs will act as the key for the data for that specific trip. The data will be. Mean % of occupation. (Mean % of occupation per stop(indicating direction)) Mean %of delays and Mean minutes of delay in case it exists.
        new_bus_json_data[f"{bus}"][f"{data_inputs[0]}-{data_inputs[1]}"] = {"Mean_Occupation_Percentage": trip_occupation_percentage, "Direction": direction, "Mean_time_of_trip": mean_time_of_trip, "Percentage_sims_delay_departure": total_percentage_of_delays_per_trip, "Mean_Minutes_departure_delay" : mean_delay_in_minutes_per_trip}




#Time between stops.
json_stops_info_toA["times_stops"] = {}
counter = 0
for stop in stops_list_toA[1:]:

    json_stops_info_toA["times_stops"][f"{stops_list_toA[counter]}-{stop}"] = {}

    #first, select all the instances from results which contains the stop_code 'stop'
    data_stop = joint_dataframe_results[joint_dataframe_results["stop_num"] == stop]
    data_stop = data_stop[data_stop["direction"] == "toA"]

    #the general value would be the mean of that column
    json_stops_info_toA["times_stops"][f"{stops_list_toA[counter]}-{stop}"]["general_time"] = data_stop["time_between_stops"].mean()

    json_stops_info_toA["times_stops"][f"{stops_list_toA[counter]}-{stop}"]["time_per_timeslot"] = {}

    times_known = known_times_stops_toA[known_times_stops_toA["StopsCodes"] == f"{stops_list_toA[counter]}-{stop}"]


    #now iterate for every timeslot to obtain its val
    for timeslot in timeslots:
        data_stop_per_timeslot = data_stop[data_stop["timeslot"] == timeslot]["time_between_stops"]
        json_stops_info_toA["times_stops"][f"{stops_list_toA[counter]}-{stop}"]["time_per_timeslot"][timeslot] = data_stop_per_timeslot.mean()

        normal_sim, var_sim, mean_sim, std_sim = is_normal_simplified(data_stop_per_timeslot.tolist())

        mean_known = times_known[f"Mean_{timeslot}"].iloc[0]
        std_known = times_known[f"Std_{timeslot}"].iloc[0]

        compute_and_save_distribution_comparasion_times(mean_sim, std_sim, mean_known, std_known, "to_A", [stops_list_toA[counter],stop], timeslot)



    counter += 1


json_stops_info_toB["times_stops"] = {}
counter = 0
for stop in stops_list_toB[1:]:

    json_stops_info_toB["times_stops"][f"{stops_list_toB[counter]}-{stop}"] = {}

    #first, select all the instances from results which contains the stop_code 'stop'
    data_stop = joint_dataframe_results[joint_dataframe_results["stop_num"] == stop]
    data_stop = data_stop[data_stop["direction"] == "toB"]

    #the general value would be the mean of that column
    json_stops_info_toB["times_stops"][f"{stops_list_toB[counter]}-{stop}"]["general_time"] = data_stop["time_between_stops"].mean()

    json_stops_info_toB["times_stops"][f"{stops_list_toB[counter]}-{stop}"]["time_per_timeslot"] = {}

    times_known = known_times_stops_toB[known_times_stops_toB["StopsCodes"] == f"{stops_list_toB[counter]}-{stop}"]


    #now iterate for every timeslot to obtain its val
    for timeslot in timeslots:
        data_stop_per_timeslot = data_stop[data_stop["timeslot"] == timeslot]["time_between_stops"]
        json_stops_info_toB["times_stops"][f"{stops_list_toB[counter]}-{stop}"]["time_per_timeslot"][timeslot] = data_stop_per_timeslot.mean()

        normal_sim, var_sim, mean_sim, std_sim = is_normal_simplified(data_stop_per_timeslot.tolist())

        mean_known = times_known[f"Mean_{timeslot}"].iloc[0]
        std_known = times_known[f"Std_{timeslot}"].iloc[0]

        compute_and_save_distribution_comparasion_times(mean_sim, std_sim, mean_known, std_known, "to_B", [stops_list_toB[counter],stop], timeslot)
    counter += 1



/tmp/ipykernel_35083/1202440814.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return central_moment(x, 3) / central_moment(x, 2)**(3/2)
/tmp/ipykernel_35083/1202440814.py:38: RuntimeWarning: invalid value encountered in scalar divide
  return (central_moment(x, 4) / central_moment(x, 2) ** 2) -3
/tmp/ipykernel_35083/2269347810.py:279: RuntimeWarning: Mean of empty slice
  mean_time_awaital_passenger_timeslotted = np.nanmean(
/tmp/ipykernel_35083/2269347810.py:279: RuntimeWarning: Mean of empty slice
  mean_time_awaital_passenger_timeslotted = np.nanmean(
/tmp/ipykernel_35083/1202440814.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return central_moment(x, 3) / central_moment(x, 2)**(3/2)
/tmp/ipykernel_35083/1202440814.py:38: RuntimeWarning: invalid value encountered in scalar divide
  return (central_moment(x, 4) / central_moment(x, 2) ** 2) -3
/tmp/ipykernel_35083/2269347810.py:343: RuntimeWarning: Mean of empty slice
  mean_time_awaital_pa

In [17]:

def sanitize_for_json(obj):
    """Recursively replace NaN/Inf with None for valid JSON."""
    if isinstance(obj, dict):
        return {k: sanitize_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [sanitize_for_json(v) for v in obj]
    elif isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    elif isinstance(obj, (np.floating, np.integer)):
        val = float(obj)
        return None if math.isnan(val) or math.isinf(val) else val
    return obj


with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/general_info.json", "w") as f:
    json.dump(sanitize_for_json(new_bus_json_data), f,indent=4)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/json_output_general_info.json", "w") as f:
    json.dump(sanitize_for_json(json_output_general_info), f,indent=4)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/general_info_toA.json", "w") as f:
    json.dump(sanitize_for_json(json_stops_info_toA), f,indent=4)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/general_info_toB.json", "w") as f:
    json.dump(sanitize_for_json(json_stops_info_toB), f,indent=4)